# Sprint 6: Policy dashboard

The signed-in app follows the wireframes: a choropleth map with a parcel card, then an XAI summary (SHAP bars, feature values, prediction path). It opens only after Firebase email/password sign-in and a second factor.

1. A sign-in wall. Parcel coordinates are not in that page.
2. Choropleth map: one marker per building, coloured by the Sprint 5 action, with a parcel card for flood probability, deprivation class, and a link to the XAI summary.
3. XAI summary for the selected parcel. Feature names are the five GraphSAGE inputs (footprint area, distances to water, sanitation, and road, roof material), not the placeholder labels drawn on the wireframe.
4. A banner stating the river-corridor lift was 0.929, so the layer is not a validated flood-hazard map.

Enable this in the Firebase console before opening the server: Email/Password sign-in, and TOTP multi-factor authentication (Identity Platform). Copy the web app `apiKey`, `authDomain`, and `projectId` into the environment. Do not commit them.


## 0. Setup

Open this notebook in Colab if you are joining the Drive policy table to the graphs. The map server itself is local: Colab's localhost is not a public URL.


In [ ]:
%%bash
BRANCH=cursor/sprint6-dashboard-d3fc
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git
if [ -d /content/nairobi-risk-mapping/.git ]; then
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || git fetch origin main
  git checkout "$BRANCH" 2>/dev/null || git checkout main
  git pull --ff-only origin "$BRANCH" 2>/dev/null || git pull --ff-only origin main
else
  git clone "$REPO" /content/nairobi-risk-mapping
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || true
  git checkout "$BRANCH" 2>/dev/null || git checkout main
fi
cd /content/nairobi-risk-mapping
git rev-parse --abbrev-ref HEAD
ls src/dashboard/app.py


In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt


## 1. Gate and explanations

A password-only token is refused. A token that records `sign_in_second_factor` is accepted. The checks below use claim dicts, not a live Firebase project.


In [ ]:
import torch
from torch_geometric.data import Data

from src.dashboard.auth_gate import mfa_satisfied
from src.models.attention_unet_model import AttentionUNetModel
from src.models.graphsage_model import GraphSAGEModel

assert not mfa_satisfied({"firebase": {"sign_in_provider": "password"}})
assert mfa_satisfied({"firebase": {"sign_in_second_factor": "totp"}})

graph = Data(
    x=torch.randn(8, 5),
    edge_index=torch.tensor([[0, 1, 2], [1, 2, 0]], dtype=torch.long),
    y=torch.randint(0, 3, (8,)),
)
sage = GraphSAGEModel(in_channels=5, hidden_channels=8, num_classes=3, dropout=0.0)
report = sage.explain(graph, node_index=0)
print("SHAP features:", report["ranked_features"])
print("attributions:", [round(float(v), 4) for v in report["attributions"]])

unet = AttentionUNetModel(in_channels=9, base_channels=4, device="cpu")
heat = unet.explain(torch.rand(1, 9, 32, 32))
print("Grad-CAM", tuple(heat.shape), "range", round(float(heat.min()), 3), round(float(heat.max()), 3))


## 2. Serve the map

Copy `dashboard/firebase_config.example.json` to a local file that is not committed, fill in the web app values, and export them before starting the server:

```bash
export FIREBASE_API_KEY=...
export FIREBASE_AUTH_DOMAIN=your-project.firebaseapp.com
export FIREBASE_PROJECT_ID=your-project
python -m src.dashboard.app --table data/policy_table.npz --graphs data/graphs
```

`--table` is the Sprint 5 `policy_table.npz`. `--graphs` is the directory of `{settlement}_graph.pt` files that now carry `data.pos`. The process joins those centroids in settlement-key order and listens on port 8765.

First sign-in with no authenticator enrolls one, then signs you out. Sign in again and enter the current code. Only that second session can fetch `/api/map`.


In [ ]:
from pathlib import Path

import numpy as np

from src.dashboard.map_view import build_policy_map, join_graph_positions
from src.models.fusion import load_policy_table
from src.models.multi_settlement import load_settlement_graphs

table_path = Path("data/policy_table.npz")
graphs_dir = Path("data/graphs")
if table_path.exists() and graphs_dir.exists():
    table = load_policy_table(table_path)
    keys = [str(k) for k in table["settlement_keys"]]
    graphs = load_settlement_graphs(graphs_dir, keys)
    records = join_graph_positions(table, graphs)
    print("joined parcels", records["positions"].shape[0], "crs", records["pos_crs"])
    preview = build_policy_map(
        records["positions"][:2],
        records["action_index"][:2],
        flood_proba=np.asarray(records["flood_proba"])[:2],
        deprivation_class=np.asarray(records["deprivation_class"])[:2],
        settlement_index=np.asarray(records["settlement_index"])[:2],
        settlement_keys=keys,
        pos_crs=records["pos_crs"],
    )
    print("map preview centre", preview.location)
    print("Start the gated server with: python -m src.dashboard.app --table", table_path, "--graphs", graphs_dir)
else:
    print("No local policy table. On Colab, copy checkpoints/sprint5/policy_table.npz to data/policy_table.npz")
    print("and keep data/graphs from Sprint 5, then run python -m src.dashboard.app")


## Done

- Unauthenticated responses do not contain parcel coordinates. Tests cover a password-only token (403) and a TOTP token (200).
- Quote the map with the river-corridor caveat. The official 37-zone lift is still outstanding.
